# 01. Downstream Class-Head Grad-CAM Explainability Visualization

This notebook implements **Grad-CAM (Gradient-weighted Class Activation Mapping)** for downstream fine-tuned FILIP diagnostic models (PTB-XL 23-Subclass classification).

### Overview
1. Selects a target sample by `TARGET_STUDY_ID` (or picks a random test sample if empty).
2. Loads the fine-tuned FILIP downstream 23-subclass model checkpoint (`outputs/filip/vit_large_ptbxl_sub_report_align_adapt_100/best.pt`).
3. **Ranked Predictions**: Prints all 23 subclass diagnostic predictions ranked from highest to lowest probability until **all Ground Truth (GT) classes** for the sample are reached.
4. **Multi-Class Grad-CAM**: Generates and overlays **Warm Red vs. Transparent** Grad-CAM heatmaps for both the **highest predicted class** and all **ground-truth (GT) classes**.

In [ ]:
import os
import sys
import yaml
import random
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from matplotlib.colors import LinearSegmentedColormap

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from filip.model.filip_ecg_model import FILIPECGModel
from filip.data.dataset import ECGImageDataset

# Mapping dictionary for all 23 PTB-XL Subclass diagnostic acronyms
SUBCLASS_EXPANSIONS = {
    'AMI': 'anterior myocardial infarction',
    'LAFB/LPFB': 'left anterior or posterior fascicular block',
    'LVH': 'left ventricular hypertrophy',
    'STTC': 'ST-T segment changes',
    'IMI': 'inferior myocardial infarction',
    'SEHYP': 'septal hypertrophy',
    'CRBBB': 'complete right bundle branch block',
    'WPW': 'Wolff-Parkinson-White syndrome',
    'LAO/LAE': 'left atrial overload or enlargement',
    'NORM': 'normal sinus rhythm',
    'ISC': 'ischemic ST-T changes',
    'AVB': 'atrioventricular block',
    'RAO/RAE': 'right atrial overload or enlargement',
    'LMI': 'lateral myocardial infarction',
    'ISCI': 'ischemic infarction',
    'ISCA': 'ischemic anterior changes',
    'NST': 'non-specific ST-T changes',
    'CLBBB': 'complete left bundle branch block',
    'ILBBB': 'incomplete left bundle branch block',
    'IRBBB': 'incomplete right bundle branch block',
    'PMI': 'posterior myocardial infarction',
    'RVH': 'right ventricular hypertrophy',
    'IVCD': 'intraventricular conduction delay'
}

# Custom Warm Red vs. Transparent Colormap (alpha 0.0 at 0, 0.75 warm red at peak)
colors = [(1.0, 0.0, 0.0, 0.0), (1.0, 0.15, 0.0, 0.75)]
warm_red_cmap = LinearSegmentedColormap.from_list('WarmRedTransparent', colors)

## 1. Load Model & Adaptation Configuration (23 Subclasses)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to 23 PTB-XL Subclass adaptation config & checkpoint
# Default to newest ViT-Large report-aligned model checkpoint
checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'vit_large_ptbxl_sub_report_align_adapt_100')
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'ptbxl_sub_report_align_adapt_100')
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = os.path.join(PROJECT_ROOT, 'outputs', 'filip', 'ptbxl_sub_adapt_100')

checkpoint_path = os.path.join(checkpoint_dir, 'best.pt')
if not os.path.exists(checkpoint_path):
    checkpoint_path = os.path.join(checkpoint_dir, 'best_model.pt')

# Select matching config file corresponding to checkpoint architecture
if 'vit_large' in checkpoint_dir:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'report_alignment_adapt_vit_large', 'ptbxl_sub_adapt.yaml')
elif 'report_align' in checkpoint_dir:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'report_alignment_adapt', 'ptbxl_sub_adapt.yaml')
else:
    config_path = os.path.join(PROJECT_ROOT, 'filip', 'configs', 'ptbxl_sub_adapt.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"Loading config from: {config_path}")

# Initialize Model
model = FILIPECGModel(config).to(device)
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    sd = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(sd, strict=False)
    print(f"Successfully loaded downstream 23-subclass model weights from: {checkpoint_path}")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}. Using base initialized model.")

## 2. Select Sample (Specified Study ID or Random Selection)

In [ ]:
from filip.visualization.diagnostic_utils import ExpandToSquare
# Set TARGET_STUDY_ID to a specific Study ID (e.g. "20149_hr"), or leave empty "" for random selection
TARGET_STUDY_ID = ""

data_root = os.path.join(PROJECT_ROOT, config.get('data_root', 'data/ptbxl_sub_class/'))
image_size = config.get('model', {}).get('image_size', 224)
dataset_name = config.get('dataset_name', 'ptbxl_sub')

transform = transforms.Compose([
    ExpandToSquare(background_color=(255, 255, 255)),
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])

test_dataset = ECGImageDataset(data_root=data_root, split='test', dataset_name=dataset_name, transform=transform)

# Select target record index
selected_idx = None
if TARGET_STUDY_ID.strip():
    for idx, record in enumerate(test_dataset.records):
        if str(record.get('study_id')) == TARGET_STUDY_ID.strip():
            selected_idx = idx
            break
    if selected_idx is None:
        print(f"Study ID '{TARGET_STUDY_ID}' not found in test set. Defaulting to random selection.")

if selected_idx is None:
    selected_idx = random.randint(0, len(test_dataset) - 1)

sample = test_dataset[selected_idx]
input_tensor = sample['images'].unsqueeze(0).to(device)
study_id = sample['sample_ids']
diagnosis_targets = sample['diagnosis_targets']

# Identify Ground Truth Labels
gt_labels = []
if diagnosis_targets is not None:
    for i, target_val in enumerate(diagnosis_targets):
        if target_val > 0.5 and i < len(test_dataset.diagnosis_list):
            gt_labels.append(test_dataset.diagnosis_list[i])

print("=" * 70)
print(f"Selected Sample Index: {selected_idx} / {len(test_dataset)}")
print(f"Selected Study ID:     {study_id}")
print(f"Ground Truth Diagnoses: {gt_labels if gt_labels else 'None / Normal'}")
print("=" * 70)

## 3. Forward Pass & Ranked Class Predictions (Printed Until All GT Classes Reached)

In [ ]:
from filip.visualization.diagnostic_utils import ExpandToSquare
model.eval()
with torch.no_grad():
    outputs = model(input_tensor)
    diag_logits = outputs['diagnosis_logits'] # [1, Num_Classes]
    probs = torch.sigmoid(diag_logits)[0]

num_model_classes = diag_logits.shape[1]
active_diag_list = test_dataset.diagnosis_list[:num_model_classes]

# Rank predictions from high to low across all 23 subclasses
sorted_indices = torch.argsort(probs, descending=True).cpu().numpy()

found_gt_set = set()
total_gt_set = set(gt_labels)

print("=" * 70)
print(f"Class Predictions (Ranked High to Low across all {num_model_classes} subclasses):")
print("=" * 70)
for rank, idx in enumerate(sorted_indices, 1):
    if idx < len(active_diag_list):
        cls_code = active_diag_list[idx]
        prob_val = probs[idx].item()
        expansion = SUBCLASS_EXPANSIONS.get(cls_code, cls_code)
        
        is_gt = False
        if cls_code in total_gt_set:
            is_gt = True
            found_gt_set.add(cls_code)
            
        gt_tag = " [GROUND TRUTH]" if is_gt else ""
        print(f"{rank:2d}. {cls_code:<12} ({prob_val:6.2%}) - {expansion}{gt_tag}")
        
        # Stop once all GT classes have been listed (or list all 23 if no GT labels)
        if total_gt_set and found_gt_set == total_gt_set:
            print(f"--> Reached all {len(total_gt_set)} Ground Truth classes at Rank {rank}.")
            break
print("=" * 70)

# Determine target classes for Grad-CAM (Highest + Ground Truth classes if not highest)
top_cls_code = active_diag_list[sorted_indices[0]]
target_classes = [top_cls_code] + [c for c in gt_labels if c in active_diag_list and c != top_cls_code]
print(f"Target Visualization Classes (Highest + GT): {target_classes}")

## 4. Compute & Render Multi-Class Warm Red Grad-CAM Heatmaps

In [ ]:
from filip.visualization.diagnostic_utils import ExpandToSquare
patch_size = config.get('model', {}).get('patch_size', 14)
grid_size = image_size // patch_size
cam_results = {}

# Compute Grad-CAM for each target class
for cls_code in target_classes:
    if cls_code in test_dataset.diagnosis_list:
        cls_idx = test_dataset.diagnosis_list.index(cls_code)
        if cls_idx < diag_logits.shape[1]:
            model.zero_grad()
            patch_features = model.vision_encoder(input_tensor) # [1, P, H]
            patch_features.retain_grad()
            
            if hasattr(model, 'diagnosis_head'):
                if getattr(model, 'diagnosis_from_features', False):
                    feature_logits = model.feature_alignment_head(patch_features)[0]
                    diag_logits_eval = model.diagnosis_head(feature_logits)
                else:
                    pooled_image = patch_features.mean(dim=1)
                    pooled_image = model.dropout(pooled_image)
                    diag_logits_eval = model.diagnosis_head(pooled_image)
            
            cls_score = diag_logits_eval[0, cls_idx]
            cls_score.backward()
            
            patch_grads = patch_features.grad.cpu()
            patch_acts = patch_features.detach().cpu()
            
            weights = torch.mean(patch_grads, dim=1, keepdim=True) # [1, 1, H]
            cam = torch.sum(weights * patch_acts, dim=-1)[0]       # [P]
            cam = F.relu(cam)
            
            cam_grid = cam.reshape(grid_size, grid_size).numpy()
            cam_grid = (cam_grid - cam_grid.min()) / (cam_grid.max() - cam_grid.min() + 1e-8)
            
            prob_val = torch.sigmoid(diag_logits_eval)[0, cls_idx].item()
            cam_results[cls_code] = (cam_grid, prob_val)

# Load raw ECG image for visualization overlay
raw_img_path = os.path.join(data_root, 'images', f"{study_id}-0.png")
if not os.path.exists(raw_img_path):
    raw_img_path = os.path.join(data_root, 'images', f"{study_id}.png")
raw_image = Image.open(raw_img_path).convert('RGB').resize((image_size, image_size))

# Render Subplot Grid: Original ECG + Grad-CAM Heatmaps for Target Classes
num_plots = 1 + len(cam_results)
fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))
if num_plots == 1:
    axes = [axes]

# Subplot 1: Original ECG Image
axes[0].imshow(raw_image)
axes[0].set_title(f"Original ECG (Study ID: {study_id})\nGT: {gt_labels if gt_labels else 'Normal'}", fontsize=10)
axes[0].axis('off')

# Subplots 2+: Grad-CAM Heatmaps per Target Class
for i, (cls_code, (cam_grid, prob_val)) in enumerate(cam_results.items(), start=1):
    expansion = SUBCLASS_EXPANSIONS.get(cls_code, cls_code)
    is_gt_tag = " [GT]" if cls_code in gt_labels else " [Top Pred]"
    
    axes[i].imshow(raw_image)
    im = axes[i].imshow(cam_grid, cmap=warm_red_cmap, extent=(0, image_size, image_size, 0))
    axes[i].set_title(f"Grad-CAM: {cls_code}{is_gt_tag} ({prob_val:.1%})\n({expansion})", fontsize=10)
    axes[i].axis('off')
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04, label="Grad-CAM Activation")

plt.tight_layout()
plt.show()
#03579_hr